In [1]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import precision_recall_curve, roc_curve
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [3]:
test_data = np.load("../turbine_data/CART2/turbine.z01.b0.20191102.175456.startup.shutdown/train.npy")

test_labels = np.load("../turbine_data/CART2/turbine.z01.b0.20191102.175456.startup.shutdown/test_labels.npy")
test_labels = 1-2*test_labels

In [4]:
# data = np.load("../turbine_data/t2/train.npy")

# labels = np.load("../turbine_data/t2/test_labels.npy")
# labels = 1-2*labels

In [1]:
precisions = []
recalls = []
fprs = []
fscores = []

clf = IsolationForest(max_samples=128, random_state=0)
clf.fit(test_data)
scores = clf.decision_function(test_data)

for cont in np.arange(0.01,1,0.02):
    print(cont)
    
    preds = np.where(scores<np.percentile(scores, cont*100), -1, 1)
    
    # [tp,tn,fp,fn]
    stats = sum(np.array([(label==-1 and pred==-1), (label==1 and pred==1), (label==1 and pred==-1), (label==-1 and pred==1)]) for label, pred in zip(test_labels, preds))
    print("TP", stats[0])
    print("TN", stats[1])
    print("FP", stats[2])
    print("FN", stats[3])
    # print()
    precisions.append(stats[0]/(stats[0]+stats[2]))
    recalls.append(stats[0]/(stats[0]+stats[3]))
    fprs.append(stats[2]/(stats[1]+stats[2]))
    fscores.append(2*precisions[-1]*recalls[-1]/(precisions[-1]+recalls[-1]))

In [2]:
for i in range(len(fprs)):
    print(precisions[i], recalls[i])

In [3]:
import re
import pandas as pd

def parse_anomaly_file(file_path):
    with open(file_path, 'r') as f:
        content = f.read()

    # Regex to find each block of data
    # It accounts for the "Anomaly ratio" and the four confusion matrix components
    pattern = r"Anomaly ratio:\s*([\d.]+).*?TP:\s*(\d+).*?FP:\s*(\d+).*?TN:\s*(\d+).*?FN:\s*(\d+)"
    
    # findall returns a list of tuples: (ratio, tp, fp, tn, fn)
    matches = re.findall(pattern, content, re.DOTALL)
    
    data = []
    for match in matches:
        ratio = float(match[0])
        tp = int(match[1])
        fp = int(match[2])
        tn = int(match[3])
        fn = int(match[4])
        
        # Calculate Metrics
        # We use max(1, ...) or conditional checks to avoid division by zero
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        
        data.append({
            'Anomaly_Ratio': ratio,
            'TP': tp,
            'FP': fp,
            'TN': tn,
            'FN': fn,
            'Precision': round(precision, 4),
            'Recall': round(recall, 4),
            'FPR': round(fpr, 4)
        })

    return pd.DataFrame(data)

results = parse_anomaly_file("/home/mcwave/code/physics/results/turbines/CART2_modattn.txt")
precisions_2 = results["Precision"]
recalls_2 = results["Recall"]
fprs_2 = results["FPR"]

In [1]:
plt.plot(fprs, recalls)
# plt.plot(recalls_2, precisions_2)
plt.xlim([0,1])
plt.ylim([0,1])
plt.grid()
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.legend(loc="upper left")
plt.show()

In [ ]:
# np.save("../results/IForest_CARE/precs.npy", precisions)
# np.save("../results/IForest_CARE/recs.npy", recalls)
# np.save("../results/IForest_CARE/fprs.npy", fprs)

In [ ]:
# np.save("../results/AnomalyTransformer/precs_20191102_modattn.npy", precisions_2)
# np.save("../results/AnomalyTransformer/recs_20191102_modattn.npy", recalls_2)
# np.save("../results/AnomalyTransformer/fprs_20191102_modattn.npy", fprs_2)